<a href="https://colab.research.google.com/github/Qureshiii/PyTorch-Learning-Journey/blob/main/10_Recurrent_Neural_Networks_RNN_Architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from keras.datasets import imdb

In [ ]:
(X_train, y_train), (X_test, y_test) = imdb.load_data()

In [ ]:
# 1. Hyperparameters
vocab_size = 10000
embedding_dim = 32
hidden_dim = 32
sequence_length = 50
batch_size = 64
epochs = 5

In [ ]:
# --- [NOTE] ---
# In a real setup, you would load X_train, y_train here.
# Assuming X_train shape is (Num_Samples, 50) and y_train is (Num_Samples,)
# Let's generate dummy tensors matching your padded Keras data shapes to demonstrate:
X_train_tensor = torch.randint(0, vocab_size, (25000, sequence_length))
# Using .float() method directly on the tensor
y_train_tensor = torch.randint(0, 2, (25000, 1)).float()

X_test_tensor = torch.randint(0, vocab_size, (25000, sequence_length))
# Using .float() method directly on the tensor
y_test_tensor = torch.randint(0, 2, (25000, 1)).float()

# Create PyTorch DataLoaders to handle batching smoothly
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [ ]:


# 2. Define the Model Architecture
class IMDBRNNModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(IMDBRNNModel, self).__init__()
        # Keras Embedding -> PyTorch Embedding
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)
        # Keras SimpleRNN -> PyTorch RNN (batch_first=True makes shape match Keras behavior)
        self.rnn = nn.RNN(input_size=embedding_dim, hidden_size=hidden_dim, batch_first=True)
        # Keras Dense(1) -> PyTorch Linear(hidden_dim, 1)
        self.fc = nn.Linear(hidden_dim, 1)
        # Activation function
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x shape: [batch_size, sequence_length]
        embedded = self.embedding(x)  # shape: [batch_size, sequence_length, embedding_dim]

        out, h_n = self.rnn(embedded) # out shape: [batch_size, sequence_length, hidden_dim]

        # return_sequences=False behavior: grab only the output of the very last time-step
        last_timestep_out = out[:, -1, :] # shape: [batch_size, hidden_dim]

        dense_out = self.fc(last_timestep_out)
        return self.sigmoid(dense_out)

# Initialize model
model = IMDBRNNModel(vocab_size, embedding_dim, hidden_dim)

# 3. Setup Loss Function and Optimizer
criterion = nn.BCELoss() # Binary Cross Entropy Loss
optimizer = optim.Adam(model.parameters(), lr=0.001)


# 4. The Training Loop (Explicitly written in PyTorch style)
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch_x, batch_y in train_loader:
        # Zero gradients out from last step
        optimizer.zero_grad()

        # Forward Pass
        predictions = model(batch_x)
        loss = criterion(predictions, batch_y)

        # Backward Pass & Weights Update
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")


Epoch 1/5 - Loss: 0.6968
Epoch 2/5 - Loss: 0.6914
Epoch 3/5 - Loss: 0.6848
Epoch 4/5 - Loss: 0.6713
Epoch 5/5 - Loss: 0.6466
